# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library as defined by its Croissant schema.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the full dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display dataset-level metadata
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
if hasattr(dataset.metadata, "keywords"):
    print("Keywords:", ', '.join(dataset.metadata.keywords))
if hasattr(dataset.metadata, "datePublished"):
    print("Date Published:", dataset.metadata.datePublished)
if hasattr(dataset.metadata, "spatialCoverage"):
    print("Spatial Coverage:", dataset.metadata.spatialCoverage)
if hasattr(dataset.metadata, "personalSensitiveInformation"):
    print("Personal Sensitive Information:", ', '.join(dataset.metadata.personalSensitiveInformation))
if hasattr(dataset.metadata, "license"):
    print("License:", dataset.metadata.license)

## 2. Data Overview
Review the available record sets, fields, and their `@id`s as declared in the dataset schema. Each entity in Croissant (record set, field, column) is referenced by its `@id`. We'll enumerate record sets and provide their `@id`, field `@id`s, and data file sources.

In [ ]:
# List all record sets and their fields by `@id`
from pprint import pprint

record_sets_info = []
for record_set in dataset.record_sets:
    rs_info = {}
    rs_info['@id'] = record_set.id_
    rs_info['name'] = getattr(record_set, 'name', '')
    rs_info['fields'] = []
    for field in record_set.fields:
        rs_info['fields'].append({'@id': field.id_, 'name': getattr(field, 'name', '')})
    rs_info['columns'] = []
    for column in getattr(record_set, 'columns', []):
        rs_info['columns'].append({'@id': column.id_, 'name': getattr(column, 'name', '')})
    rs_info['source'] = getattr(record_set, 'source', None)
    record_sets_info.append(rs_info)

if len(record_sets_info) == 0:
    print("No record sets discovered in 'recordSet'. Inspecting dataset distribution to infer record sets...")

# Sometimes Croissant schemas don't set up record sets directly, need to infer them from files/distributions
if not dataset.record_sets:
    print("Dataset's 'recordSets' is empty. Listing distributions:")
    if hasattr(dataset.metadata, "distribution"):
        for d in dataset.metadata.distribution:
            if hasattr(d, "contentUrl"):
                print("Distribution:", d['@id'], "--", d['contentUrl'])
            else:
                print("Distribution @id:", d['@id'])
else:
    print("Record Sets\n===========")
    for rs in record_sets_info:
        print(f"@id: {rs['@id']}")
        if rs['name']: print(f"  name: {rs['name']}")
        if rs['fields']:
            print(f"  Fields:")
            for field in rs['fields']:
                print(f"    @id: {field['@id']} (name: {field['name']})")
        if rs['columns']:
            print(f"  Columns:")
            for col in rs['columns']:
                print(f"    @id: {col['@id']} (name: {col['name']})")
        if rs['source']:
            print(f"  Source: {rs['source']}")
        print("")

## 3. Data Extraction
Load data from each discovered record set into a pandas DataFrame for further analysis. Reference each entity by its `@id` as per the schema.

In [ ]:
# List of discovered record_set @id's
record_set_ids = [rs["@id"] for rs in record_sets_info]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for record set {record_set_id}")
    print(f"Columns available: {df.columns.tolist()}\n")

# If only one record set, select it for demonstration; else prompt the user to choose
if record_set_ids:
    selected_record_set_id = record_set_ids[0]  # choose the first record set for this example
    print(f"\nFirst 5 records from record set '{selected_record_set_id}':")
    display(dataframes[selected_record_set_id].head())
else:
    print("No data extracted from record sets.")

## 4. Exploratory Data Analysis (EDA)
Process and analyze the DataFrame. We'll select a numeric field (referenced by its `@id`) for further filtering, normalization, and grouping, assuming such a field is present in the record set. Adjust field IDs as per your earlier overview.

In [ ]:
# Find a numeric field for analysis (edit as appropriate for your dataset structure)
import numpy as np

df = dataframes[selected_record_set_id]
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # pick first numeric field by column name (assumed as @id)
    print(f"Selected numeric field for analysis: {numeric_field_id}")
else:
    print("No numeric fields found in the data.")
    # Optionally, try to cast columns to numeric if they're stored as string
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        raise ValueError('No numeric fields available for EDA.')
    numeric_field_id = numeric_fields[0]

# Filter: keep only records where the numeric field exceeds its 10th percentile as a threshold.
threshold = df[numeric_field_id].quantile(0.1)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold}")
display(filtered_df.head())

# Normalize the numeric field (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
    filtered_df[numeric_field_id].std(ddof=0)
)
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a categorical field (choose the first string/object column distinct from the numeric field)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == object:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize the distribution of the numeric field, and its relationship to the chosen categorical group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram of the numeric field
plt.figure(figsize=(6,4))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field exists, plot boxplot/grouped means
if group_field_id:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using the Croissant schema, explored its contents via record set and field `@id`s, extracted the data into pandas DataFrames, and performed basic exploratory data analysis and visualization. Future directions may include modeling, advanced statistical analysis, or domain-specific interpretation in context of adoption predictors of indigenous and modern knowledge in rangeland management.